# 🧬 PyHyB Tutorial: Building Hybrid Molecular Structures

This notebook walks through the complete PyHyB workflow:

1. **Understanding inputs** — What are `.xyz` and `.cif` files?
2. **Building a hybrid** — Deposit a molecule on a substrate
3. **Inspecting results** — Examine the combined structure
4. **Tuning parameters** — Placement, rotation, collision threshold
5. **Reading the manifest** — Provenance and reproducibility

---

## 1. Setup & Imports

Make sure PyHyB is installed (`pip install -e .` from the repo root).

In [ ]:
import sys
from pathlib import Path

# Ensure PyHyB is importable
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyhyb.core.runner import run_build_hybrid
from pyhyb.tools.builder import HybridBuilder, HybridConfig
from pyhyb.info import __version__, PROGRAM_NAME

print(f'{PROGRAM_NAME} v{__version__} loaded successfully!')

## 2. Understanding Input Files

PyHyB takes two inputs:

| Input | Format | Description |
|-------|--------|-------------|
| **Material** | `.xyz` | The molecule to deposit (e.g., glycine, crown ether) |
| **Substrate** | `.cif` | The periodic substrate (e.g., graphene) |

Let's examine the example data:

In [ ]:
from ase.io import read

DATA = PROJECT_ROOT / 'example' / 'data'

# Read the glycine molecule
glycine = read(DATA / 'glycine.xyz')
print(f'Glycine: {len(glycine)} atoms')
print(f'  Elements: {set(glycine.symbols)}')
print(f'  Bounding box (Å):')
for axis, name in enumerate(['X', 'Y', 'Z']):
    lo = glycine.positions[:, axis].min()
    hi = glycine.positions[:, axis].max()
    print(f'    {name}: {lo:.2f} — {hi:.2f} (span: {hi-lo:.2f})')

# Read the graphene substrate
graphene = read(DATA / 'graphene_7x7.cif')
print(f'\nGraphene 7×7: {len(graphene)} atoms')
print(f'  Cell dimensions: {graphene.cell.lengths()}')

## 3. Building Your First Hybrid Structure

The simplest way to build a hybrid is using `run_build_hybrid()`:

In [ ]:
import tempfile

# Use a temporary directory for outputs
with tempfile.TemporaryDirectory() as tmpdir:
    msg = run_build_hybrid(
        material_path=str(DATA / 'glycine.xyz'),
        substrate_path=str(DATA / 'graphene_7x7.cif'),
        output_dir=tmpdir,
        placement='surface',     # Place on top of substrate
        distance=3.0,            # 3 Å above the surface
        collision_threshold=1.5, # Min 1.5 Å between atoms
    )
    print(msg)

## 4. Inspecting the Result

Let's build again and examine the combined structure in detail:

In [ ]:
output_dir = PROJECT_ROOT / 'example' / 'output' / 'tutorial'

msg = run_build_hybrid(
    material_path=str(DATA / 'glycine.xyz'),
    substrate_path=str(DATA / 'graphene_7x7.cif'),
    output_dir=str(output_dir),
    placement='surface',
    distance=3.0,
    collision_threshold=1.5,
)

# Read back the generated structure
hybrid = read(output_dir / 'hybrid_structure.cif')

print(f'Hybrid structure: {len(hybrid)} atoms')
print(f'  Elements: {dict(zip(*zip(*[(s, 1) for s in hybrid.symbols])))}')
print(f'  Cell Z-height: {hybrid.cell[2, 2]:.2f} Å')
print(f'  Z range: {hybrid.positions[:, 2].min():.2f} — {hybrid.positions[:, 2].max():.2f} Å')

## 5. Tuning Parameters

### Placement modes

| Mode | Description |
|------|-------------|
| `'surface'` | Bottom of molecule placed `distance` Å above the highest local substrate atom |
| `'center'` | Centre of molecule placed at Z = cell_height / 2 (intercalation) |

### Rotation

You can pre-rotate the molecule around its centre of mass using Euler angles `[rx, ry, rz]` in degrees.

In [ ]:
import tempfile

# Compare center vs surface placement
for mode in ['center', 'surface']:
    with tempfile.TemporaryDirectory() as tmpdir:
        run_build_hybrid(
            material_path=str(DATA / 'glycine.xyz'),
            substrate_path=str(DATA / 'graphene_7x7.cif'),
            output_dir=tmpdir,
            placement=mode,
            distance=3.0,
            collision_threshold=1.5,
        )
        h = read(Path(tmpdir) / 'hybrid_structure.cif')
        z_min = h.positions[:, 2].min()
        z_max = h.positions[:, 2].max()
        print(f'{mode:>8s}: Z range = [{z_min:.1f}, {z_max:.1f}] Å')

In [ ]:
# Build with a 45° rotation around Z
with tempfile.TemporaryDirectory() as tmpdir:
    run_build_hybrid(
        material_path=str(DATA / 'glycine.xyz'),
        substrate_path=str(DATA / 'graphene_7x7.cif'),
        output_dir=tmpdir,
        placement='surface',
        distance=3.0,
        collision_threshold=1.5,
        rotation=[0, 0, 45],  # 45° around Z-axis
    )
    h = read(Path(tmpdir) / 'hybrid_structure.cif')
    print(f'Rotated hybrid: {len(h)} atoms')
    print(f'  Z range: [{h.positions[:, 2].min():.1f}, {h.positions[:, 2].max():.1f}] Å')

## 6. Build Manifest (Provenance)

Every build writes a `*_manifest.json` alongside the output files.
This captures all inputs, parameters, and metadata for reproducibility.

In [ ]:
import json

manifest_path = output_dir / 'hybrid_structure_manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

for key, value in manifest.items():
    print(f'  {key}: {value}')

## 7. Advanced: Using the Builder API Directly

For more control, use `HybridConfig` + `HybridBuilder` directly:

In [ ]:
config = HybridConfig(
    material_path=DATA / 'crown_ether.xyz',
    substrate_path=DATA / 'graphene_7x7.cif',
    outdir=output_dir / 'crown_ether_advanced',
    placement='surface',
    distance=2.5,
    collision_threshold=1.8,
    rotation=[0, 0, 30],
)

builder = HybridBuilder(config)
hybrid = builder.build()

# Write outputs
cif, gen, xyz = builder.write_outputs(hybrid)
manifest = builder.write_manifest(hybrid)

print(f'Crown ether hybrid: {len(hybrid)} atoms')
print(f'  CIF: {cif}')
print(f'  Manifest: {manifest}')

## 8. Default Parameters

All parameter defaults are centralised in `HybridConfig.defaults()`:

In [ ]:
from pyhyb.tools.builder import HybridConfig

defaults = HybridConfig.defaults()
for key, value in defaults.items():
    print(f'  {key}: {value}')

## 📝 Summary

| Feature | How to use |
|---------|------------|
| Quick build | `run_build_hybrid(material, substrate, ...)` |
| Advanced build | `HybridConfig` + `HybridBuilder` |
| Placement modes | `'center'` (intercalation) or `'surface'` (deposition) |
| Rotation | `rotation=[rx, ry, rz]` in degrees |
| Collision safety | `collision_threshold` in Å |
| Provenance | `*_manifest.json` auto-generated |
| Web GUI | `streamlit run app.py` |
| Interactive menu | `python launcher.py` |

---

For more details, see the [README.md](../README.md) or the Streamlit Web GUI.